# MaIA_SDCI Final Project

**Sistemas de Decision y Control Inteligente (SDCI)**  
**Universidad de los Andes, 2026**

Copyright (c) 2026 Leffer Trochez

---

Computes performance metrics for one simulation run.


In [ ]:
import numpy as np
import pandas as pd


def _trapz(values, time):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(values, time)
    return np.trapz(values, time)


def compute_metrics(tTemperature, T_ref, T_meas, tPower, Q_heat, controllerID):
    """Compute the project performance metrics."""

    # METRIC SETTINGS
    settlingPercentage = 0.02
    steadyStateFraction = 0.10

    # INPUT VALIDATION
    tTemperature = np.asarray(tTemperature, dtype=float).reshape(-1)
    tPower = np.asarray(tPower, dtype=float).reshape(-1)
    T_ref = np.asarray(T_ref, dtype=float)
    T_meas = np.asarray(T_meas, dtype=float)
    Q_heat = np.asarray(Q_heat, dtype=float)

    if T_ref.shape[0] != tTemperature.size:
        raise ValueError("The number of rows in T_ref must match the number of temperature time samples.")
    if T_meas.shape[0] != tTemperature.size:
        raise ValueError("The number of rows in T_meas must match the number of temperature time samples.")
    if Q_heat.shape[0] != tPower.size:
        raise ValueError("The number of rows in Q_heat must match the number of power time samples.")
    if T_ref.shape[1] < 4 or T_meas.shape[1] < 4 or Q_heat.shape[1] < 4:
        raise ValueError("T_ref, T_meas and Q_heat must contain the common four-channel project interface.")
    if tTemperature.size < 2 or tPower.size < 2:
        raise ValueError("At least two samples are required to compute metrics.")

    # ACTIVE ROOMS
    if int(controllerID) == 4:
        activeChannels = [0]
        roomNames = ["Room_5"]
    else:
        activeChannels = [0, 1, 2, 3]
        roomNames = ["Room_1", "Room_2", "Room_3", "Room_4"]

    rows = []
    for room_name, channel in zip(roomNames, activeChannels):
        tref = T_ref[:, channel]
        tmeas = T_meas[:, channel]
        validTemperature = np.isfinite(tTemperature) & np.isfinite(tref) & np.isfinite(tmeas)
        tT = tTemperature[validTemperature]
        tref = tref[validTemperature]
        tmeas = tmeas[validTemperature]

        metrics = {
            "Room": room_name,
            "RMSE_degC": np.nan,
            "IAE_degC_s": np.nan,
            "ISE_degC2_s": np.nan,
            "SteadyStateError_degC": np.nan,
            "SettlingTime_s": np.nan,
            "Overshoot_degC": np.nan,
            "Energy_kWh": np.nan,
            "PeakPower_W": np.nan,
        }

        if tT.size >= 2:
            error = tref - tmeas
            metrics["RMSE_degC"] = float(np.sqrt(np.mean(error ** 2)))
            metrics["IAE_degC_s"] = float(_trapz(np.abs(error), tT))
            metrics["ISE_degC2_s"] = float(_trapz(error ** 2, tT))

            nSteady = max(1, int(np.ceil(steadyStateFraction * error.size)))
            metrics["SteadyStateError_degC"] = float(np.mean(error[-nSteady:]))

            referenceFinal = float(tref[-1])
            settlingTolerance = settlingPercentage * abs(referenceFinal)
            outsideBand = np.abs(error) > settlingTolerance
            violation_indices = np.flatnonzero(outsideBand)
            if violation_indices.size == 0:
                metrics["SettlingTime_s"] = float(tT[0])
            else:
                lastViolation = int(violation_indices[-1])
                if lastViolation < tT.size - 1:
                    metrics["SettlingTime_s"] = float(tT[lastViolation + 1])

            metrics["Overshoot_degC"] = float(max(0.0, np.max(tmeas - tref)))

        q = Q_heat[:, channel]
        validPower = np.isfinite(tPower) & np.isfinite(q)
        tQ = tPower[validPower]
        q = q[validPower]
        if tQ.size >= 2:
            metrics["Energy_kWh"] = float(_trapz(np.maximum(q, 0.0), tQ) / 3.6e6)
            metrics["PeakPower_W"] = float(np.max(q))

        rows.append(metrics)

    return pd.DataFrame(rows)
